In [1]:
# Re-connect ke Blob Storage
from azure.storage.blob import BlobServiceClient
import io

STORAGE_ACCOUNT = "stfinalertai"
STORAGE_KEY = "YOUR_AZURE_STORAGE_KEY"
CONTAINER = "finalert-data"

blob_service = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
    credential=STORAGE_KEY
)
container_client = blob_service.get_container_client(CONTAINER)
print("✅ Reconnected ke Blob Storage!")

✅ Reconnected ke Blob Storage!


In [2]:
# Cell 1 — Load master panel
from azure.storage.blob import BlobServiceClient
import pandas as pd
import numpy as np
import io
import warnings
warnings.filterwarnings('ignore')

STORAGE_ACCOUNT = "stfinalertai"
STORAGE_KEY = "YOUR_AZURE_STORAGE_KEY"
CONTAINER = "finalert-data"

blob_service = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
    credential=STORAGE_KEY
)
container_client = blob_service.get_container_client(CONTAINER)

# Load master panel
blob = container_client.get_blob_client("cleaned/master_panel.parquet")
data = blob.download_blob().readall()
df = pd.read_parquet(io.BytesIO(data))
df['periode'] = pd.to_datetime(df['periode'], errors='coerce')
df = df.dropna(subset=['periode'])
df = df[df['periode'].dt.year >= 2021].sort_values(['provinsi','periode']).reset_index(drop=True)

print(f"✅ Master panel loaded: {df.shape}")
print(f"   Provinsi: {df['provinsi'].nunique()}")
print(f"   Periode : {df['periode'].min().date()} s/d {df['periode'].max().date()}")

✅ Master panel loaded: (2280, 20)
   Provinsi: 38
   Periode : 2021-01-01 s/d 2025-12-01


In [3]:
# Cell 2 — Feature Engineering: Lag & Rolling Features
print("Membuat lag & rolling features...")

df = df.sort_values(['provinsi','periode']).reset_index(drop=True)

# Lag features TWP90 (1, 3, 6 bulan)
for lag in [1, 3, 6]:
    df[f'twp90_lag{lag}'] = df.groupby('provinsi')['twp90_pct'].shift(lag)

# Rolling mean TWP90 (3, 6 bulan)
for window in [3, 6]:
    df[f'twp90_roll{window}'] = df.groupby('provinsi')['twp90_pct'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )

# Outstanding per rekening
df['outstanding_per_rekening'] = df['outstanding'] / df['n_rekening'].replace(0, np.nan)

# Outstanding growth MoM
df['outstanding_growth_mom'] = df.groupby('provinsi')['outstanding'].pct_change()

# Interaction term: internet × DPK (paradoks inklusi)
df['internet_x_literasi'] = df['penetrasi_internet'] * df['dpk']

# BI Rate lag 3 bulan & perubahan
df['bi_rate_lag3'] = df.groupby('provinsi')['bi_rate'].shift(3)
df['bi_rate_change'] = df['bi_rate'].diff()

# Dummy variables
df['dummy_jawa'] = df['provinsi'].isin([
    'Banten','Dki Jakarta','Jawa Barat','Jawa Tengah',
    'Daerah Istimewa Yogyakarta','Jawa Timur'
]).astype(int)

df['dummy_island'] = df['provinsi'].map(lambda p: 
    'jawa' if p in ['Banten','Dki Jakarta','Jawa Barat','Jawa Tengah','Daerah Istimewa Yogyakarta','Jawa Timur']
    else 'sumatera' if p in ['Aceh','Sumatera Utara','Sumatera Barat','Riau','Jambi',
                              'Sumatera Selatan','Bengkulu','Lampung','Kepulauan Bangka Belitung','Kepulauan Riau']
    else 'kalimantan' if 'Kalimantan' in p
    else 'sulawesi' if 'Sulawesi' in p or p in ['Gorontalo']
    else 'papua' if 'Papua' in p
    else 'ntt_ntb' if p in ['Nusa Tenggara Barat','Nusa Tenggara Timur']
    else 'maluku' if 'Maluku' in p
    else 'bali_nusa' if p in ['Bali']
    else 'lainnya'
)

# Quarter dummy
df['dummy_q1'] = (df['periode'].dt.month <= 3).astype(int)

# TKD per kapita (normalisasi)
df['total_tkd_per_kapita'] = df['total_tkd_miliar'] / df['n_rekening'].replace(0, np.nan)

print(f"✅ Feature engineering selesai!")
print(f"   Total kolom: {len(df.columns)}")
print(f"\nKolom baru:")
new_cols = ['twp90_lag1','twp90_lag3','twp90_lag6','twp90_roll3','twp90_roll6',
            'outstanding_per_rekening','outstanding_growth_mom','internet_x_literasi',
            'bi_rate_lag3','bi_rate_change','dummy_jawa','dummy_island','dummy_q1',
            'total_tkd_per_kapita']
print(df[new_cols].describe().round(3))

Membuat lag & rolling features...
✅ Feature engineering selesai!
   Total kolom: 34

Kolom baru:
       twp90_lag1  twp90_lag3  twp90_lag6  twp90_roll3  twp90_roll6  \
count    2242.000    2166.000    2052.000     2242.000     2242.000   
mean        1.762       1.749       1.740        1.741        1.715   
std         0.975       0.952       0.959        0.936        0.915   
min         0.030       0.030       0.030        0.127        0.158   
25%         1.170       1.160       1.148        1.187        1.157   
50%         1.640       1.640       1.630        1.615        1.607   
75%         2.230       2.217       2.206        2.210        2.187   
max        11.560       7.670       7.670        7.480        7.027   

       outstanding_per_rekening  outstanding_growth_mom  internet_x_literasi  \
count                  2280.000                2242.000             1184.000   
mean                      0.207                  29.302           190263.446   
std                    

In [4]:
# Cell TAMBAHAN — Imputasi provinsi Papua baru & Kaltara
prov_missing = [
    'Papua Selatan', 'Papua Tengah', 'Papua Pegunungan',
    'Papua Barat Daya', 'Kalimantan Utara'
]

bank_cols = ['dpk','ldr','npl_ratio','kredit_umkm','rasio_umkm','jml_kc_bank']

print("Sebelum imputasi:")
print(f"  Baris NaN perbankan: {df[bank_cols].isnull().any(axis=1).sum()}")

for col in bank_cols:
    # Imputasi dengan median provinsi yang datanya ada
    median_val = df[~df['provinsi'].isin(prov_missing)][col].median()
    df.loc[df['provinsi'].isin(prov_missing), col] = \
        df.loc[df['provinsi'].isin(prov_missing), col].fillna(median_val)
    print(f"  ✅ {col}: diimputasi dengan median = {median_val:.4f}")

print(f"\nSetelah imputasi:")
print(f"  Baris NaN perbankan: {df[bank_cols].isnull().any(axis=1).sum()}")
print(f"  Provinsi yang diimputasi: {prov_missing}")

Sebelum imputasi:
  Baris NaN perbankan: 712
  ✅ dpk: diimputasi dengan median = 52392.2300
  ✅ ldr: diimputasi dengan median = 1.0084
  ✅ npl_ratio: diimputasi dengan median = 0.0218
  ✅ kredit_umkm: diimputasi dengan median = 22436.2150
  ✅ rasio_umkm: diimputasi dengan median = 0.3810
  ✅ jml_kc_bank: diimputasi dengan median = 64.0000

Setelah imputasi:
  Baris NaN perbankan: 412
  Provinsi yang diimputasi: ['Papua Selatan', 'Papua Tengah', 'Papua Pegunungan', 'Papua Barat Daya', 'Kalimantan Utara']


In [5]:
# Jalankan ulang Cell 3 — Simpan master_features yang sudah difix
buffer = io.BytesIO()
df.to_parquet(buffer, index=False)
buffer.seek(0)

blob_client = container_client.get_blob_client("features/master_features.parquet")
blob_client.upload_blob(buffer, overwrite=True)

print(f"✅ master_features.parquet (updated) berhasil diupload!")
print(f"   Shape: {df.shape}")
print(f"   NaN perbankan tersisa: {df[bank_cols].isnull().any(axis=1).sum()}")

✅ master_features.parquet (updated) berhasil diupload!
   Shape: (2280, 34)
   NaN perbankan tersisa: 412


In [6]:
# Cek hasil interpolasi — apakah nilai bulanan masuk akal?
print("="*60)
print("CEK KUALITAS INTERPOLASI")
print("="*60)

# Cek PDRB — harusnya naik smooth per bulan
print("\n📊 PDRB Per Kapita — Aceh (sample 12 bulan pertama):")
sample = df[df['provinsi']=='Aceh'][['periode','pdrb_per_kapita']].head(12)
print(sample.to_string(index=False))

# Cek TPT — harusnya interpolasi antara Feb & Agu
print("\n📊 TPT — Aceh (sample 12 bulan pertama):")
sample2 = df[df['provinsi']=='Aceh'][['periode','tpt']].head(12)
print(sample2.to_string(index=False))

# Cek Internet — harusnya naik smooth per bulan
print("\n📊 Penetrasi Internet — Aceh (sample 24 bulan):")
sample3 = df[df['provinsi']=='Aceh'][['periode','penetrasi_internet']].head(24)
print(sample3.to_string(index=False))

# Cek TKD
print("\n📊 TKD Miliar — Aceh (sample 12 bulan):")
sample4 = df[df['provinsi']=='Aceh'][['periode','total_tkd_miliar']].head(12)
print(sample4.to_string(index=False))

# Cek DPK
print("\n📊 DPK — Aceh (sample 12 bulan):")
sample5 = df[df['provinsi']=='Aceh'][['periode','dpk']].head(12)
print(sample5.to_string(index=False))

CEK KUALITAS INTERPOLASI

📊 PDRB Per Kapita — Aceh (sample 12 bulan pertama):
   periode  pdrb_per_kapita
2021-01-01     3.467400e+07
2021-02-01     3.501508e+07
2021-03-01     3.535617e+07
2021-04-01     3.569725e+07
2021-05-01     3.603833e+07
2021-06-01     3.637942e+07
2021-07-01     3.672050e+07
2021-08-01     3.706158e+07
2021-09-01     3.740267e+07
2021-10-01     3.774375e+07
2021-11-01     3.808483e+07
2021-12-01     3.842592e+07

📊 TPT — Aceh (sample 12 bulan pertama):
   periode   tpt
2021-01-01   NaN
2021-02-01 6.300
2021-03-01 6.300
2021-04-01 6.300
2021-05-01 6.300
2021-06-01 6.300
2021-07-01 6.300
2021-08-01 6.300
2021-09-01 6.245
2021-10-01 6.190
2021-11-01 6.135
2021-12-01 6.080

📊 Penetrasi Internet — Aceh (sample 24 bulan):
   periode  penetrasi_internet
2021-01-01                 NaN
2021-02-01                 NaN
2021-03-01                 NaN
2021-04-01                 NaN
2021-05-01                 NaN
2021-06-01                 NaN
2021-07-01                 NaN


In [7]:
# Cek September hilang
print("Cek periode unik — apakah September ada?")
periods = df['periode'].dt.strftime('%Y-%m').unique()
periods_sorted = sorted(periods)
for p in periods_sorted[:15]:
    print(f"  {p}")

Cek periode unik — apakah September ada?
  2021-01
  2021-02
  2021-03
  2021-04
  2021-05
  2021-06
  2021-07
  2021-08
  2021-09
  2021-10
  2021-11
  2021-12
  2022-01
  2022-02
  2022-03


In [8]:
# Cek semua September yang hilang
import pandas as pd
all_periods = pd.date_range('2021-01-01', '2025-12-01', freq='MS')
existing_periods = df['periode'].unique()

missing = [p for p in all_periods if p not in existing_periods]
print(f"Total periode yang harusnya ada: {len(all_periods)}")
print(f"Total periode yang ada: {len(existing_periods)}")
print(f"\nPeriode yang HILANG:")
for m in missing:
    print(f"  ❌ {m.strftime('%Y-%m')}")

Total periode yang harusnya ada: 60
Total periode yang ada: 60

Periode yang HILANG:
